# Temporal Holdout Data from 2022 to 2025. STRING PPI Data

Downloads: from https://string-db.org/cgi/download and from https://string-db.org/mapping_files/uniprot/

```
wget https://stringdb-downloads.org/download/protein.links.detailed.v12.0.txt.gz
wget https://stringdb-downloads.org/download/protein.info.v12.0.txt.gz

wget https://stringdb-downloads.org/download/protein.aliases.v12.0.txt.gz
wget https://string-db.org/mapping_files/uniprot/all_organisms.uniprot_2_string.tsv

pigz -d protein.info.v12.0.txt.gz
pigz -d protein.aliases.v12.0.txt.gz

grep "UniProt_AC" protein.aliases.v12.0.txt > protein.aliases.v12.0.uniprot_ac.txt
```

# Initial Analysis

In [1]:
from datasets import load_dataset

# Load the train split
temp_holdout_ext_data = load_dataset(
    "wanglab/cafa5", 'temporal_holdout_ext_2022_2025_reasoning'
)

print(temp_holdout_ext_data)

DatasetDict({
    test: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'go_bp_created', 'go_mf_created', 'go_cc_created', 'date_added', 'last_modified', 'length', 'structure_path', 'interpro_ids', 'interpro_location'],
        num_rows: 17812
    })
})


In [2]:
# Convert HuggingFace Datasets to Polars DataFrames
import pandas as pd
import polars as pl

# Convert train and test datasets to pandas DataFrames, then to polars
temp_holdout_ext_data = temp_holdout_ext_data['test'].to_pandas()

temp_holdout_ext_data = pl.from_pandas(temp_holdout_ext_data)

# sanity check
temp_holdout_ext_data.head()

protein_id,protein_names,protein_function,organism,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc,go_bp_created,go_mf_created,go_cc_created,date_added,last_modified,length,structure_path,interpro_ids,interpro_location
str,str,str,str,str,str,str,str,str,str,f64,f64,f64,str,str,i64,str,list[str],str
"""P0DO85""","""Strychnine-10-hydroxylase""","""Monooxygenase involved in the …","""Strychnos nux-vomica (Poison n…","""Membrane; Single-pass membrane…","""MEFSLLYIHTAILGLISLFLILHFVFWRLK…","""['GO:0008150' 'GO:0008152' 'GO…","""['GO:0008150' 'GO:0008152' 'GO…","""['GO:0003674' 'GO:0003824' 'GO…",null,2.0240628e7,2.0240628e7,null,"""2024-11-27""","""2025-06-18""",524,"""AF-P0DO85-F1-model_v6.pdb""","[""IPR050651"", ""IPR017972"", … ""IPR036396""]","""{""IPR036396"": [33, 518], ""IPR0…"
"""P0DO87""","""Strychnine-11-hydroxylase""","""Monooxygenase involved in the …","""Strychnos nux-vomica (Poison n…","""Membrane; Single-pass membrane…","""MHSAMSFLLLFSLCFLIHCFVFLLIKKKKA…","""['GO:0008150' 'GO:0008152' 'GO…","""['GO:0008150' 'GO:0008152' 'GO…","""['GO:0003674' 'GO:0003824' 'GO…",null,2.0240628e7,2.0240628e7,null,"""2024-11-27""","""2025-06-18""",508,"""AF-P0DO87-F1-model_v6.pdb""","[""IPR017972"", ""IPR036396"", … ""IPR002401""]","""{""IPR036396"": [23, 500], ""IPR0…"
"""P46077""","""14-3-3-like protein GF14 phi""","""Is associated with a DNA bindi…","""Arabidopsis thaliana (Mouse-ea…","""Nucleus. Cytoplasm. Note=Trans…","""MAAPPASSSAREEFVYLAKLAEQAERYEEM…","""['GO:0007154' 'GO:0007165' 'GO…","""['GO:0007154' 'GO:0007165' 'GO…",null,null,2.0080421e7,null,null,"""1995-11-01""","""2025-06-18""",267,"""AF-P46077-F1-model_v6.pdb""","[""IPR023410"", ""IPR036815"", … ""IPR000308""]","""{""IPR036815"": [2, 254], ""IPR00…"
"""P48348""","""14-3-3-like protein G-BOX fact…","""Is associated with a DNA bindi…","""Arabidopsis thaliana (Mouse-ea…","""Nucleus. Cytoplasm. Note=Trans…","""MATTLSRDQYVYMAKLAEQAERYEEMVQFM…","""['GO:0005575' 'GO:0005622' 'GO…",null,null,"""['GO:0005575' 'GO:0005622' 'GO…",null,null,2.0221025e7,"""1996-02-01""","""2025-06-18""",248,"""AF-P48348-F1-model_v6.pdb""","[""IPR023410"", ""IPR036815"", … ""IPR000308""]","""{""IPR036815"": [1, 248], ""IPR00…"
"""Q96299""","""14-3-3-like protein GF14 mu""","""Is associated with a DNA bindi…","""Arabidopsis thaliana (Mouse-ea…","""Nucleus. Cytoplasm. Note=Trans…","""MGSGKERDTFVYLAKLSEQAERYEEMVESM…","""['GO:0007275' 'GO:0008150' 'GO…","""['GO:0007275' 'GO:0008150' 'GO…",null,null,2.0120604e7,null,null,"""1998-07-15""","""2025-06-18""",263,"""AF-Q96299-F1-model_v6.pdb""","[""IPR023410"", ""IPR036815"", … ""IPR000308""]","""{""IPR036815"": [1, 247], ""IPR00…"


Load the uniprot to string conversion table (from programmatic access of the uniprot mapping tool)

In [3]:
# load uniprot to string data

mapping_path = "string/all_organisms.uniprot_2_string.tsv"
cols = ["species", "uniprot_ac|uniprot_id", "string_id", "identity", "bit_score"]

# Load and clean, skipping the bad header
uniprot_string_map_df = pl.read_csv(
    mapping_path,
    separator="\t",
    comment_prefix="#",
    has_header=False,
    new_columns=cols,
    ignore_errors=True
)

# Split the combined column into accession and ID, then drop the original
uniprot_string_map_df = (
    uniprot_string_map_df
    .with_columns(
        pl.col("uniprot_ac|uniprot_id")
          .str.split_exact("|", 1)
          .struct
          .rename_fields(["uniprot_ac", "uniprot_id"])
          .alias("split")
    )
    .unnest("split")
    .drop("uniprot_ac|uniprot_id")
)


In [4]:
uniprot_string_map_df.head(5)

species,string_id,identity,bit_score,uniprot_ac,uniprot_id
i64,str,str,str,str,str
5874,"""5874.P16019""","""Src""","""Src""","""P16019""","""HSP70_THEAN"""
5874,"""5874.P25781""","""Src""","""Src""","""P25781""","""CYSP_THEAN"""
5874,"""5874.Q26671""","""Src""","""Src""","""Q26671""","""CDC2H_THEAN"""
5874,"""5874.Q27399""","""Src""","""Src""","""Q27399""","""Q27399_THEAN"""
5874,"""5874.Q37679""","""Src""","""Src""","""Q37679""","""COX3_THEAN"""


Reads the string/name table from raw text to polars

In [5]:
import polars as pl

# --- Configuration fallback: only set if not already defined ---
if 'protein_info_path' not in globals():
    protein_info_path = "string/protein.info.v12.0.txt"

# Define the actual column names (strip the leading '#')
cols = ["string_protein_id", "preferred_name", "protein_size", "annotation"]

# --- Load the uncompressed protein.info file, using our header names ---
protein_info_df = pl.read_csv(
    protein_info_path,
    separator="\t",
    has_header=False,   # file's first line isn't a usable header
    skip_rows=1,        # skip the '#...' header line
    new_columns=cols,    # apply our cleaned column names
    ignore_errors=True,
)

# Preview columns & first few rows
print("Columns:", protein_info_df.columns)
print(protein_info_df.head(5))


Columns: ['string_protein_id', 'preferred_name', 'protein_size', 'annotation']
shape: (5, 4)
┌───────────────────┬────────────────┬──────────────┬─────────────────────────────────┐
│ string_protein_id ┆ preferred_name ┆ protein_size ┆ annotation                      │
│ ---               ┆ ---            ┆ ---          ┆ ---                             │
│ str               ┆ str            ┆ i64          ┆ str                             │
╞═══════════════════╪════════════════╪══════════════╪═════════════════════════════════╡
│ 23.BEL05_00025    ┆ BEL05_00025    ┆ 429          ┆ Hypothetical protein; Incomple… │
│ 23.BEL05_00030    ┆ OEG73659.1     ┆ 203          ┆ 2-keto-4-pentenoate hydratase;… │
│ 23.BEL05_00035    ┆ OEG73660.1     ┆ 1140         ┆ Hybrid sensor histidine kinase… │
│ 23.BEL05_00040    ┆ acsA           ┆ 650          ┆ acetate--CoA ligase; Catalyzes… │
│ 23.BEL05_00045    ┆ OEG73662.1     ┆ 579          ┆ ATP-dependent helicase; Derive… │
└───────────────────┴──────

In [6]:
protein_info_df.shape

(59309604, 4)

Brings the string ids into the cafa5 dataset, matching based on the uniprot id

In [7]:
# join uniprot cafa5 data with string id

import polars as pl

# 2) Prepare the mapping: only uniprot_ac → string_id
map_pl = uniprot_string_map_df.select(["uniprot_ac", "string_id"])

# 3) Join on protein_id ↔ uniprot_ac
joined = temp_holdout_ext_data.join(
    map_pl,
    left_on="protein_id",
    right_on="uniprot_ac",
    how="left"
)

temp_holdout_ext_data = joined.select(
    temp_holdout_ext_data.columns + ["string_id"]
)

# 5) Quick check
print(temp_holdout_ext_data.head(10))


shape: (10, 20)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ protein_i ┆ protein_n ┆ protein_f ┆ organism  ┆ … ┆ structure ┆ interpro_ ┆ interpro_ ┆ string_i │
│ d         ┆ ames      ┆ unction   ┆ ---       ┆   ┆ _path     ┆ ids       ┆ location  ┆ d        │
│ ---       ┆ ---       ┆ ---       ┆ str       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│ str       ┆ str       ┆ str       ┆           ┆   ┆ str       ┆ list[str] ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ P0DO85    ┆ Strychnin ┆ Monooxyge ┆ Strychnos ┆ … ┆ AF-P0DO85 ┆ ["IPR0506 ┆ {"IPR0363 ┆ null     │
│           ┆ e-10-hydr ┆ nase      ┆ nux-vomic ┆   ┆ -F1-model ┆ 51", "IPR ┆ 96": [33, ┆          │
│           ┆ oxylase   ┆ involved  ┆ a (Poison ┆   ┆ _v6.pdb   ┆ 017972",  ┆ 518],     ┆          │
│           ┆           ┆ in the …  ┆ n…        ┆   ┆           ┆ … "…     

Convert the String data file from txt to a parquet file

In [ ]:
# This cell efficiently converts the STRING data file to a Parquet file using DuckDB.
# MEMORY OPTIMIZED VERSION for large files (190GB+) on systems with limited RAM

import duckdb
import os
import psutil

# Paths
input_path = 'string/protein.links.detailed.v12.0.txt.gz'
output_dir = 'string/parquet_output_duckdb'
output_path = os.path.join(output_dir, 'protein_links_space_delimited.parquet')

# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)

# Check available memory and disk space
available_memory_gb = 256
file_size_gb = os.path.getsize(input_path) / (1024**3)
print(f"File size: {file_size_gb:.1f} GB")
print(f"Available memory: {available_memory_gb:.1f} GB")

print("Starting memory-optimized conversion with DuckDB...")

# MEMORY OPTIMIZATION SETTINGS
# Limit memory usage to ~60% of available RAM to leave room for OS and other processes
max_memory_gb = max(1, int(available_memory_gb * 0.6))
print(f"Setting DuckDB memory limit to {max_memory_gb} GB")

# Reduce thread count for memory-constrained environments
thread_count = 16  # Use fewer threads to reduce memory pressure
print(f"Using {thread_count} threads")

# Connect to DuckDB with optimized configuration
# Using a temporary database file allows DuckDB to spill to disk when needed
temp_db_path = os.path.join(output_dir, 'temp_conversion.duckdb')
config = {
    'memory_limit': f'{max_memory_gb}GB',
    'threads': thread_count,
    'temp_directory': '/tmp',
    'preserve_insertion_order': False,
    'enable_external_access': True
}

con = duckdb.connect(temp_db_path, config=config)

try:
    
    # Process the file with memory-efficient settings
    print("Converting file (this may take longer but uses less memory)...")
    con.execute(f"""
        COPY (
            SELECT *
            FROM read_csv(
                '{input_path}', 
                header=True, 
                delim=' ', 
                compression='gzip',
                sample_size=100000  -- Smaller sample for schema detection to save memory
            )
        ) TO '{output_path}' (
            FORMAT 'PARQUET', 
            CODEC 'ZSTD',
            ROW_GROUP_SIZE 100000  -- Smaller row groups for better memory management
        );
    """)
    
    print(f"Successfully converted the entire file to {output_path}")
    
finally:
    # Close the connection and clean up temporary database
    con.close()
    if os.path.exists(temp_db_path):
        os.remove(temp_db_path)
        print("Cleaned up temporary database file")

File size: 189.6 GB
Available memory: 256.0 GB
Starting memory-optimized conversion with DuckDB...
Setting DuckDB memory limit to 153 GB
Using 16 threads
Converting file (this may take longer but uses less memory)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [8]:
# SYSTEM REQUIREMENTS CHECK - Run this before attempting conversion
import os
import shutil
import psutil

def check_system_requirements(input_path, output_dir):
    """Check if system has enough resources for the conversion"""
    
    print("=== SYSTEM REQUIREMENTS CHECK ===")
    
    # File size
    if os.path.exists(input_path):
        file_size_gb = os.path.getsize(input_path) / (1024**3)
        print(f"Input file size: {file_size_gb:.1f} GB")
    else:
        print(f"WARNING: Input file not found: {input_path}")
        return False
    
    # Available memory
    memory = psutil.virtual_memory()
    available_memory_gb = memory.available / (1024**3)
    total_memory_gb = memory.total / (1024**3)
    print(f"Available RAM: {available_memory_gb:.1f} GB / {total_memory_gb:.1f} GB")
    
    # Available disk space
    disk_usage = shutil.disk_usage(output_dir)
    available_disk_gb = disk_usage.free / (1024**3)
    print(f"Available disk space: {available_disk_gb:.1f} GB")
    
    # Recommendations
    print("\n=== RECOMMENDATIONS ===")
    
    # Memory recommendation
    if available_memory_gb < file_size_gb * 0.1:  # Less than 10% of file size
        print("⚠️  WARNING: Very low available memory!")
        print("   Recommended: Use the chunked processing approach")
        print("   Set chunk_size_mb to 200-500 MB")
    elif available_memory_gb < file_size_gb * 0.3:  # Less than 30% of file size
        print("⚠️  CAUTION: Limited memory available")
        print("   Recommended: Use memory-optimized DuckDB settings")
        print("   Monitor memory usage during conversion")
    else:
        print("✅ Memory should be sufficient for DuckDB conversion")
    
    # Disk space recommendation
    estimated_parquet_size = file_size_gb * 0.3  # Rough estimate: 30% of original size
    if available_disk_gb < estimated_parquet_size * 3:  # Need 3x for temp files
        print("⚠️  WARNING: May not have enough disk space!")
        print(f"   Estimated output size: {estimated_parquet_size:.1f} GB")
        print(f"   Recommended free space: {estimated_parquet_size * 3:.1f} GB")
    else:
        print("✅ Disk space should be sufficient")
    
    print(f"\n=== SUGGESTED APPROACH ===")
    if available_memory_gb >= file_size_gb * 0.3:
        print("1. Try the memory-optimized DuckDB approach first (Cell 15)")
        print("2. If it fails, use the chunked approach (Cell 16)")
    else:
        print("1. Use the chunked processing approach (Cell 16)")
        print("2. Start with chunk_size_mb=200 and increase if stable")
    
    return True

# Run the check
input_path = 'string/protein.links.detailed.v12.0.txt.gz'
output_dir = 'string/parquet_output_duckdb'
check_system_requirements(input_path, output_dir)


=== SYSTEM REQUIREMENTS CHECK ===
Input file size: 189.6 GB
Available RAM: 455.0 GB / 503.3 GB
Available disk space: 849.8 GB

=== RECOMMENDATIONS ===
✅ Memory should be sufficient for DuckDB conversion
✅ Disk space should be sufficient

=== SUGGESTED APPROACH ===
1. Try the memory-optimized DuckDB approach first (Cell 15)
2. If it fails, use the chunked approach (Cell 16)


True

Filter string data to only include rows with confidence >70% and which are related to a cafa5 protein

Takes ~15 minutues with 64 cpus and slow SLURM disk performance, for faster performance copy protein_links_space_delimited.parquet to SLURM_TEMPDIR with above cell

In [9]:
with open("temporal-holdout-ext-string_ids.txt", "w") as f:
    for sid in temp_holdout_ext_data["string_id"].drop_nulls().unique().to_list():
        f.write(f"{sid}\n")

In [ ]:
# FIXED VERSION: Use the existing code with polars-u64-idx
# Run this after installing polars-u64-idx and restarting kernel

import polars as pl

input_parquet_path = (
    "string/parquet_output_duckdb/protein_links_space_delimited.parquet"
)

# Check if we have the large-dataset version of polars
try:
    # Test if polars supports large datasets
    print("Polars version:", pl.__version__)
    print("Checking polars build configuration...")
    
    # This will help identify if we have the right version
    dummy_large_range = range(2**33)  # Larger than 2^32
    print("✅ Python can handle large ranges")
    
except Exception as e:
    print(f"⚠️  Issue detected: {e}")

# Configuration: use conservative settings for large datasets
pl.Config.set_streaming_chunk_size(500_000)  # Smaller chunks for stability

MIN_COMBINED_SCORE = 700

# Build swissprot ID list
with open("temporal-holdout-ext-string_ids.txt") as f:
    string_ids = [line.strip() for line in f if line.strip()]
print(f"swissprot filter will use {len(string_ids)} unique string IDs")

# Use more conservative approach for large datasets
print("Setting up lazy frame with conservative settings...")

lf = pl.scan_parquet(
    input_parquet_path,
    parallel="columns",      # Use column parallelism instead of row_groups for better memory usage
    use_statistics=True,     # Enable statistics-based filtering
    low_memory=True          # Use less memory during processing
)

# Apply filters lazily
print("Applying filters...")
lf = lf.filter(pl.col("combined_score") >= MIN_COMBINED_SCORE)
lf = lf.filter(pl.col("protein1").is_in(string_ids))

# Collect with conservative settings
print("Executing query with streaming engine...")
try:
    final_df = lf.collect(engine="streaming")
    print("✅ Success! Final dataframe has shape", final_df.shape)
    
    # Save the resulting dataframe to a file
    output_path = "temporal-holdout-ext-filtered-protein-links.parquet"
    print(f"Saving dataframe to {output_path}...")
    final_df.write_parquet(output_path)
    print(f"✅ Dataframe saved successfully to {output_path}")
    print(f"   Total rows: {len(final_df):,}")
    print(f"   Columns: {final_df.columns}")
    
except Exception as e:
    print(f"❌ Error: {e}")
    print("Try running the alternative approach in the next cell")


Reads the filtered string dataframe

In [83]:
import polars as pl

# Path to the saved file
output_path = "temporal-holdout-ext-filtered-protein-links.parquet"

# Read the file
filtered_df = pl.read_parquet(output_path)

# Show shape and head
print("Shape:", filtered_df.shape)
print(filtered_df.tail())


Shape: (186613, 10)
shape: (5, 10)
┌────────────┬────────────┬────────────┬────────┬───┬───────────┬──────────┬───────────┬───────────┐
│ protein1   ┆ protein2   ┆ neighborho ┆ fusion ┆ … ┆ experimen ┆ database ┆ textminin ┆ combined_ │
│ ---        ┆ ---        ┆ od         ┆ ---    ┆   ┆ tal       ┆ ---      ┆ g         ┆ score     │
│ str        ┆ str        ┆ ---        ┆ i64    ┆   ┆ ---       ┆ i64      ┆ ---       ┆ ---       │
│            ┆            ┆ i64        ┆        ┆   ┆ i64       ┆          ┆ i64       ┆ i64       │
╞════════════╪════════════╪════════════╪════════╪═══╪═══════════╪══════════╪═══════════╪═══════════╡
│ 1801619.SA ┆ 1801619.SA ┆ 804        ┆ 0      ┆ … ┆ 0         ┆ 0        ┆ 0         ┆ 804       │
│ MN04515619 ┆ MN04515619 ┆            ┆        ┆   ┆           ┆          ┆           ┆           │
│ _11957     ┆ _11956     ┆            ┆        ┆   ┆           ┆          ┆           ┆           │
│ 1801619.SA ┆ 1801619.SA ┆ 800        ┆ 0      ┆ … ┆ 0 

Use the protein2 string id and switch it out with teh corresponding uniprot accession. use the protein.aliases.v12.0.uniprot_ac.txt file that we created above with grep

In [13]:
# --- Configuration fallback: only set if not already defined ---
protein_info_path = "string/protein.aliases.v12.0.uniprot_ac.txt"

# The file has 3 columns: #string_protein_id, alias, source
col_names = ["string_protein_id", "alias", "source"]

# Load the file, skipping header row which starts with '#'
string_mapping = pl.read_csv(
    protein_info_path,
    separator="\t",
    has_header=False,      # We're ignoring the file's first line as header
    skip_rows=1,           # Skip the commented-out header
    new_columns=col_names, # Explicitly set our true column names
    ignore_errors=True,
)

# Create dictionary mapping string_protein_id to alias (uniprot_ac)
stringid_to_uniprotac = dict(zip(
    string_mapping["string_protein_id"].to_list(),
    string_mapping["alias"].to_list()
))

# Preview the first 5 entries of the dictionary
print("First 5 string_id → uniprot_ac mappings:")
for k, v in list(stringid_to_uniprotac.items())[:5]:
    print(f"{k} -> {v}")


First 5 string_id → uniprot_ac mappings:
23.BEL05_00030 -> A0A1E5ISW8
23.BEL05_00035 -> A0A1E5ISW6
23.BEL05_00040 -> A0A1E5ISX0
23.BEL05_00045 -> A0A1E5IT53
23.BEL05_00050 -> A0A1E5IUM4


In [14]:
string_mapping

string_protein_id,alias,source
str,str,str
"""23.BEL05_00030""","""A0A1E5ISW8""","""UniProt_AC"""
"""23.BEL05_00035""","""A0A1E5ISW6""","""UniProt_AC"""
"""23.BEL05_00040""","""A0A1E5ISX0""","""UniProt_AC"""
"""23.BEL05_00045""","""A0A1E5IT53""","""UniProt_AC"""
"""23.BEL05_00050""","""A0A1E5IUM4""","""UniProt_AC"""
…,…,…
"""2656787.A0A370U463""","""A0A370U463""","""UniProt_AC"""
"""2656787.A0A370U467""","""A0A370U467""","""UniProt_AC"""
"""2656787.A0A370U473""","""A0A370U473""","""UniProt_AC"""


replace protein2 string code with protein2 name

In [84]:
import polars as pl

# --- Replace protein2 IDs with their UniProt accessions from dictionary, using pl.col().replace ---

# 1. Rename original protein2 to protein2_string
filtered_df = filtered_df.rename({"protein2": "protein2_string"})

# 2. Map protein2_string using stringid_to_uniprotac dictionary via replace
filtered_df = filtered_df.with_columns(
    pl.col("protein2_string").replace(stringid_to_uniprotac, default=None).alias("protein2")
)

# Now filtered_df.protein2 contains the uniprot accessions,
# and the original IDs are in filtered_df.protein2_string.

/tmp/ipykernel_1471203/2989869835.py:10: DeprecationWarning: The `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
  pl.col("protein2_string").replace(stringid_to_uniprotac, default=None).alias("protein2")


In [23]:
filtered_df

protein1,protein2_string,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score,protein2
str,str,i64,i64,i64,i64,i64,i64,i64,i64,str
"""882.DVU_1441""","""882.DVU_0863""",106,0,745,773,148,0,149,955,"""Q72DR6"""
"""882.DVU_1441""","""882.DVU_0862""",106,0,748,769,750,0,149,986,"""Q72DR7"""
"""882.DVU_1441""","""882.DVU_2229""",64,0,655,474,0,0,90,824,"""Q729W8"""
"""882.DVU_1441""","""882.DVU_0410""",97,0,0,794,73,0,0,812,"""Q72F08"""
"""882.DVU_1441""","""882.DVU_0048""",62,0,557,432,0,0,88,755,"""Q72G15"""
…,…,…,…,…,…,…,…,…,…,…
"""1801619.SAMN04515619_11957""","""1801619.SAMN04515619_11956""",804,0,0,0,0,0,0,804,"""A0A1I1NU16"""
"""1801619.SAMN04515619_11957""","""1801619.SAMN04515619_11955""",800,0,0,0,0,0,0,800,"""A0A1I1P074"""
"""1801619.SAMN04515619_11958""","""1801619.SAMN04515619_11956""",804,0,0,0,0,0,0,804,"""A0A1I1NU16"""


In [27]:
# Save all unique accessions from the 'protein2' column into a text file
accessions = filtered_df['protein2'].unique().to_list()
with open("temporal-holdout-ext-unique_protein2_accessions.txt", "w") as f:
    for acc in accessions:
        if acc is not None:
            f.write(str(acc) + "\n")

In [24]:
print("temporal holdout column types:")
print(temp_holdout_ext_data.schema)

print("\nFiltered DF column types:")
print(filtered_df.schema)


temporal holdout column types:
Schema({'protein_id': String, 'protein_names': String, 'protein_function': String, 'organism': String, 'subcellular_location': String, 'sequence': String, 'go_ids': String, 'go_bp': String, 'go_mf': String, 'go_cc': String, 'go_bp_created': Float64, 'go_mf_created': Float64, 'go_cc_created': Float64, 'date_added': String, 'last_modified': String, 'length': Int64, 'structure_path': String, 'interpro_ids': List(String), 'interpro_location': String, 'string_id': String})

Filtered DF column types:
Schema({'protein1': String, 'protein2_string': String, 'neighborhood': Int64, 'fusion': Int64, 'cooccurence': Int64, 'coexpression': Int64, 'experimental': Int64, 'database': Int64, 'textmining': Int64, 'combined_score': Int64, 'protein2': String})


Groups filtered string df by protein 1 and then adds column with all associated protein 2 and another with all proteins 2 along with individual scores

In [36]:
import polars as pl

grouped_df = (
    filtered_df
    .group_by("protein1")
    .agg([
        # 1. List of all protein2 values per protein1
        pl.col("protein2").implode().alias("interaction_partners")
    ])
)

print("Grouped shape:", grouped_df.shape)
print(grouped_df.head())


Grouped shape: (6801, 2)
shape: (5, 2)
┌─────────────────────────┬─────────────────────────────────┐
│ protein1                ┆ interaction_partners            │
│ ---                     ┆ ---                             │
│ str                     ┆ list[str]                       │
╞═════════════════════════╪═════════════════════════════════╡
│ 4932.YDL138W            ┆ ["Q05838", "Q03770", … "P04806… │
│ 7955.ENSDARP00000089794 ┆ ["A0A286YB53", "Q6PFU4", "A3QJ… │
│ 237561.A0A1D8PLB5       ┆ ["G1UB11", "A0A1D8PLI2", … "Q9… │
│ 284812.O94725           ┆ ["O43049", "Q9UTX1"]            │
│ 7227.FBpp0304078        ┆ ["Q9W2N7", "Q9VYG4"]            │
└─────────────────────────┴─────────────────────────────────┘


In [37]:
temp_holdout_ext_data

protein_id,protein_names,protein_function,organism,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc,go_bp_created,go_mf_created,go_cc_created,date_added,last_modified,length,structure_path,interpro_ids,interpro_location,string_id
str,str,str,str,str,str,str,str,str,str,f64,f64,f64,str,str,i64,str,list[str],str,str
"""P0DO85""","""Strychnine-10-hydroxylase""","""Monooxygenase involved in the …","""Strychnos nux-vomica (Poison n…","""Membrane; Single-pass membrane…","""MEFSLLYIHTAILGLISLFLILHFVFWRLK…","""['GO:0008150' 'GO:0008152' 'GO…","""['GO:0008150' 'GO:0008152' 'GO…","""['GO:0003674' 'GO:0003824' 'GO…",null,2.0240628e7,2.0240628e7,null,"""2024-11-27""","""2025-06-18""",524,"""AF-P0DO85-F1-model_v6.pdb""","[""IPR050651"", ""IPR017972"", … ""IPR036396""]","""{""IPR036396"": [33, 518], ""IPR0…",null
"""P0DO87""","""Strychnine-11-hydroxylase""","""Monooxygenase involved in the …","""Strychnos nux-vomica (Poison n…","""Membrane; Single-pass membrane…","""MHSAMSFLLLFSLCFLIHCFVFLLIKKKKA…","""['GO:0008150' 'GO:0008152' 'GO…","""['GO:0008150' 'GO:0008152' 'GO…","""['GO:0003674' 'GO:0003824' 'GO…",null,2.0240628e7,2.0240628e7,null,"""2024-11-27""","""2025-06-18""",508,"""AF-P0DO87-F1-model_v6.pdb""","[""IPR017972"", ""IPR036396"", … ""IPR002401""]","""{""IPR036396"": [23, 500], ""IPR0…",null
"""P46077""","""14-3-3-like protein GF14 phi""","""Is associated with a DNA bindi…","""Arabidopsis thaliana (Mouse-ea…","""Nucleus. Cytoplasm. Note=Trans…","""MAAPPASSSAREEFVYLAKLAEQAERYEEM…","""['GO:0007154' 'GO:0007165' 'GO…","""['GO:0007154' 'GO:0007165' 'GO…",null,null,2.0080421e7,null,null,"""1995-11-01""","""2025-06-18""",267,"""AF-P46077-F1-model_v6.pdb""","[""IPR023410"", ""IPR036815"", … ""IPR000308""]","""{""IPR036815"": [2, 254], ""IPR00…","""3702.P46077"""
"""P48348""","""14-3-3-like protein G-BOX fact…","""Is associated with a DNA bindi…","""Arabidopsis thaliana (Mouse-ea…","""Nucleus. Cytoplasm. Note=Trans…","""MATTLSRDQYVYMAKLAEQAERYEEMVQFM…","""['GO:0005575' 'GO:0005622' 'GO…",null,null,"""['GO:0005575' 'GO:0005622' 'GO…",null,null,2.0221025e7,"""1996-02-01""","""2025-06-18""",248,"""AF-P48348-F1-model_v6.pdb""","[""IPR023410"", ""IPR036815"", … ""IPR000308""]","""{""IPR036815"": [1, 248], ""IPR00…","""3702.P48348"""
"""Q96299""","""14-3-3-like protein GF14 mu""","""Is associated with a DNA bindi…","""Arabidopsis thaliana (Mouse-ea…","""Nucleus. Cytoplasm. Note=Trans…","""MGSGKERDTFVYLAKLSEQAERYEEMVESM…","""['GO:0007275' 'GO:0008150' 'GO…","""['GO:0007275' 'GO:0008150' 'GO…",null,null,2.0120604e7,null,null,"""1998-07-15""","""2025-06-18""",263,"""AF-Q96299-F1-model_v6.pdb""","[""IPR023410"", ""IPR036815"", … ""IPR000308""]","""{""IPR036815"": [1, 247], ""IPR00…","""3702.Q96299"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""A0A6I8RXM5""","""Mothers against decapentaplegi…",null,"""Xenopus tropicalis (Western cl…","""Cytoplasm. Nucleus""","""MFRTKRSVLVRRLWRSRAPGESGEGEGGER…","""['GO:0001702' 'GO:0003401' 'GO…","""['GO:0001702' 'GO:0003401' 'GO…",null,null,2.0180404e7,null,null,"""2020-12-02""","""2025-10-08""",357,"""AF-A0A6I8RXM5-F1-model_v6.pdb""","[""IPR036578"", ""IPR008984"", … ""IPR003619""]","""{""IPR036578"": [9, 162], ""IPR00…",null
"""F7EN59""","""Secreted frizzled-related prot…",null,"""Xenopus tropicalis (Western cl…","""Cell membrane; Multi-pass memb…","""MNSEGGIWPLLLFWVTPGILSQAPQASEYD…","""['GO:0008150' 'GO:0036268' 'GO…","""['GO:0008150' 'GO:0036268' 'GO…",null,null,2.0180404e7,null,null,"""2011-07-27""","""2025-10-08""",304,"""AF-F7EN59-F1-model_v6.pdb""","[""IPR008993"", ""IPR036790"", … ""IPR041760""]","""{""IPR036790"": [41, 165], ""IPR0…",null
"""A0A803J722""","""PC-esterase domain-containing …",null,"""Xenopus tropicalis (Western cl…",null,"""MCIYTASDRSRRDMSCFSTEHAQQLLHNKY…","""['GO:0007275' 'GO:0008150' 'GO…","""['GO:0007275' 'GO:0008150' 'GO…",null,null,2.0180409e7,null,null,"""2021-06-02""","""2025-10-08""",425,"""AF-A0A803J722-F1-model_v6.pdb""","[""IPR036514""]","""{""IPR036514"": [13, 257]}""",null


Connects the PPI data with the cafa5 data

In [38]:
merged_df = temp_holdout_ext_data.join(
    grouped_df,
    left_on="string_id",
    right_on="protein1",
    how="left"
)

# Only drop 'protein1' if it exists (e.g., when join didn't auto-drop it)
if "protein1" in merged_df.columns:
    merged_df = merged_df.drop("protein1")

print("Merged shape:", merged_df.shape)
print(merged_df.select(["string_id", "interaction_partners"]).head())


Merged shape: (17815, 21)
shape: (5, 2)
┌─────────────┬─────────────────────────────────┐
│ string_id   ┆ interaction_partners            │
│ ---         ┆ ---                             │
│ str         ┆ list[str]                       │
╞═════════════╪═════════════════════════════════╡
│ null        ┆ null                            │
│ null        ┆ null                            │
│ 3702.P46077 ┆ ["Q9LK96", "Q9LYJ6", … "Q9FUT0… │
│ 3702.P48348 ┆ ["Q9C9Q2", "Q9FZ61", … "Q8GUH5… │
│ 3702.Q96299 ┆ ["Q4V397", "Q9SRI1", … "Q9LZE1… │
└─────────────┴─────────────────────────────────┘


In [ ]:
# The percentage of rows in swissprot WITHOUT an entry in string->uniprot table
sum(merged_df['string_id'].is_null())

9881

In [ ]:
# The percent of rows in swissprot WITHOUT interaction partners
sum(merged_df['interaction_partners'].is_null())/len(merged_df)*100

61.82430536065113

In [39]:
merged_df

protein_id,protein_names,protein_function,organism,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc,go_bp_created,go_mf_created,go_cc_created,date_added,last_modified,length,structure_path,interpro_ids,interpro_location,string_id,interaction_partners
str,str,str,str,str,str,str,str,str,str,f64,f64,f64,str,str,i64,str,list[str],str,str,list[str]
"""P0DO85""","""Strychnine-10-hydroxylase""","""Monooxygenase involved in the …","""Strychnos nux-vomica (Poison n…","""Membrane; Single-pass membrane…","""MEFSLLYIHTAILGLISLFLILHFVFWRLK…","""['GO:0008150' 'GO:0008152' 'GO…","""['GO:0008150' 'GO:0008152' 'GO…","""['GO:0003674' 'GO:0003824' 'GO…",null,2.0240628e7,2.0240628e7,null,"""2024-11-27""","""2025-06-18""",524,"""AF-P0DO85-F1-model_v6.pdb""","[""IPR050651"", ""IPR017972"", … ""IPR036396""]","""{""IPR036396"": [33, 518], ""IPR0…",null,null
"""P0DO87""","""Strychnine-11-hydroxylase""","""Monooxygenase involved in the …","""Strychnos nux-vomica (Poison n…","""Membrane; Single-pass membrane…","""MHSAMSFLLLFSLCFLIHCFVFLLIKKKKA…","""['GO:0008150' 'GO:0008152' 'GO…","""['GO:0008150' 'GO:0008152' 'GO…","""['GO:0003674' 'GO:0003824' 'GO…",null,2.0240628e7,2.0240628e7,null,"""2024-11-27""","""2025-06-18""",508,"""AF-P0DO87-F1-model_v6.pdb""","[""IPR017972"", ""IPR036396"", … ""IPR002401""]","""{""IPR036396"": [23, 500], ""IPR0…",null,null
"""P46077""","""14-3-3-like protein GF14 phi""","""Is associated with a DNA bindi…","""Arabidopsis thaliana (Mouse-ea…","""Nucleus. Cytoplasm. Note=Trans…","""MAAPPASSSAREEFVYLAKLAEQAERYEEM…","""['GO:0007154' 'GO:0007165' 'GO…","""['GO:0007154' 'GO:0007165' 'GO…",null,null,2.0080421e7,null,null,"""1995-11-01""","""2025-06-18""",267,"""AF-P46077-F1-model_v6.pdb""","[""IPR023410"", ""IPR036815"", … ""IPR000308""]","""{""IPR036815"": [2, 254], ""IPR00…","""3702.P46077""","[""Q9LK96"", ""Q9LYJ6"", … ""Q9FUT0""]"
"""P48348""","""14-3-3-like protein G-BOX fact…","""Is associated with a DNA bindi…","""Arabidopsis thaliana (Mouse-ea…","""Nucleus. Cytoplasm. Note=Trans…","""MATTLSRDQYVYMAKLAEQAERYEEMVQFM…","""['GO:0005575' 'GO:0005622' 'GO…",null,null,"""['GO:0005575' 'GO:0005622' 'GO…",null,null,2.0221025e7,"""1996-02-01""","""2025-06-18""",248,"""AF-P48348-F1-model_v6.pdb""","[""IPR023410"", ""IPR036815"", … ""IPR000308""]","""{""IPR036815"": [1, 248], ""IPR00…","""3702.P48348""","[""Q9C9Q2"", ""Q9FZ61"", … ""Q8GUH5""]"
"""Q96299""","""14-3-3-like protein GF14 mu""","""Is associated with a DNA bindi…","""Arabidopsis thaliana (Mouse-ea…","""Nucleus. Cytoplasm. Note=Trans…","""MGSGKERDTFVYLAKLSEQAERYEEMVESM…","""['GO:0007275' 'GO:0008150' 'GO…","""['GO:0007275' 'GO:0008150' 'GO…",null,null,2.0120604e7,null,null,"""1998-07-15""","""2025-06-18""",263,"""AF-Q96299-F1-model_v6.pdb""","[""IPR023410"", ""IPR036815"", … ""IPR000308""]","""{""IPR036815"": [1, 247], ""IPR00…","""3702.Q96299""","[""Q4V397"", ""Q9SRI1"", … ""Q9LZE1""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""A0A6I8RXM5""","""Mothers against decapentaplegi…",null,"""Xenopus tropicalis (Western cl…","""Cytoplasm. Nucleus""","""MFRTKRSVLVRRLWRSRAPGESGEGEGGER…","""['GO:0001702' 'GO:0003401' 'GO…","""['GO:0001702' 'GO:0003401' 'GO…",null,null,2.0180404e7,null,null,"""2020-12-02""","""2025-10-08""",357,"""AF-A0A6I8RXM5-F1-model_v6.pdb""","[""IPR036578"", ""IPR008984"", … ""IPR003619""]","""{""IPR036578"": [9, 162], ""IPR00…",null,null
"""F7EN59""","""Secreted frizzled-related prot…",null,"""Xenopus tropicalis (Western cl…","""Cell membrane; Multi-pass memb…","""MNSEGGIWPLLLFWVTPGILSQAPQASEYD…","""['GO:0008150' 'GO:0036268' 'GO…","""['GO:0008150' 'GO:0036268' 'GO…",null,null,2.0180404e7,null,null,"""2011-07-27""","""2025-10-08""",304,"""AF-F7EN59-F1-model_v6.pdb""","[""IPR008993"", ""IPR036790"", … ""IPR041760""]","""{""IPR036790"": [41, 165], ""IPR0…",null,null
"""A0A803J722""","""PC-esterase domain-containing …",null,"""Xenopus tropicalis (Western cl…",null,"""MCIYTASDRSRRDMSCFSTEHAQQLLHNKY…","""['GO:0007275' 'GO:0008150' 'GO…","""['GO:0007275' 'GO:0008

get the associated protein metadata from uniprot for the interaction partners. Using the swissprot and trembl scripts from the previous notebook temporal-holdout-ext.ipynb

In [40]:
import pandas as pd

swissprot_proteins = pd.read_csv('temporal-holdout-ext-unique_protein2_accessions_swissprot.csv')

In [41]:
!cat string/trembl_logs/* | grep "No matches found in" | wc -l

23703


In [42]:
!ls string/trembl_results/ | wc -l

1604


25307 - 23703 = 1604

Verified, that I covered all the chunks and I got all the proteins that could be found with this search

In [43]:
import glob

# Match all files that start with 'parse_1279275_' and end with '.out'
files = sorted(glob.glob("string/trembl_results/results_chunk_*.csv"))

# Read and concat all, skipping the header row in each file
df = pd.concat(
    [pd.read_csv(f, sep=",", header=None, skiprows=1) for i, f in enumerate(files)],
    ignore_index=True
)

# (optional) if the first row of later chunks got concatenated as headers, fix them:
df.columns = ['Accession','Protein Name','protein_function','organism','subcellular_location', 'sequence']

print(f"✅ Combined {len(files)} files into a dataframe with {len(df):,} rows.")

✅ Combined 1604 files into a dataframe with 23,632 rows.


In [44]:
trembl_test = df
trembl_test

,Accession,Protein Name,protein_function,organism,subcellular_location,sequence
0,Q9V0P1,Threonine synthase {ECO:0000256|NCBIfam:TIGR00...,NaN,Pyrococcus abyssi (strain GE5 / Orsay).,NaN,MKCPKCGREYKAEIPPFCICGEELEITYDYSSVDTRKWKNRRPGVW...
1,Q9UY42,N-type ATP pyrophosphatase superfamily {ECO:00...,NaN,Pyrococcus abyssi (strain GE5 / Orsay).,NaN,MKCKFCSREAYIKLHYPKMYLCEEHFKEYFERKVQRTIEKYKLLKK...
2,Q9V1J8,Biotin synthase {ECO:0008006|Google:ProtNLM},NaN,Pyrococcus abyssi (strain GE5 / Orsay).,NaN,MKPKYIDKMIVDKIKNGDVVVIEYPSTFPVHEFLWDELIPTLIDEF...
3,Q9HPV3,Phytoene synthase {ECO:0000313|EMBL:AAG19764.1},NaN,Halobacterium salinarum (strain ATCC 700922 / ...,NaN,MVSPEHLRRSKAVQRRTGRTFHIATRFLPERVRYPTYVLYAFFRLT...
4,O58817,Cobalamin-independent methionine synthase MetE...,NaN,Pyrococcus horikoshii (strain ATCC 700860 / DS...,NaN,MIVPALIGSLPRPIGLAKKIELYSIGRLSEEKLEEAYRSYTRKAFE...
...,...,...,...,...,...,...
23627,Q28FD6,Zona pellucida protein D {ECO:0000313|Ensembl:...,NaN,Xenopus tropicalis (Western clawed frog) (Silu...,NaN,MNKMKYYMASWLFVGSILLLTDCISSDLIQQQNAMADLKCDNDQMT...
23628,F6YE22,Mediator of RNA polymerase II transcription su...,NaN,Xenopus tropicalis (Western clawed frog) (Silu...,NaN,MAASLSQELANLLNPQPKFRDPEDDQDEATVARVIDRFEEDGNEDD...
23629,L7N349,High affinity immunoglobulin gamma Fc receptor...,NaN,Xenopus tropicalis (Western clawed frog) (Silu...,NaN,MITGALVRPVVSFSPNWATIAKNDSITLTCTVGPTAPLSQRYSWYR...
23630,F6YR62,Family with sequence similarity 149 member A {...,NaN,Xenopus tropicalis (Western clawed frog) (Silu...,NaN,MPIAAHPYTAWRLKHRSLQQQQRQQHHTLISDTHADLVIIGKGITR...


In [45]:
protein_2_metadata = pd.concat([swissprot_proteins,trembl_test])

In [46]:
protein_2_metadata

,Accession,Protein Name,protein_function,organism,subcellular_location,sequence
0,Q9V648,Guanylate binding protein 128up {ECO:0000303|P...,Catalyzes the conversion of GTP to GDP through...,Drosophila melanogaster (Fruit fly).,NaN,MSTILEKISAIESEMARTQKNKATSAHLGLLKAKLAKLRRELISPK...
1,Q9S9Z8,14-3-3-like protein GF14 omicron,Is associated with a DNA binding complex that ...,Arabidopsis thaliana (Mouse-ear cress).,Nucleus {ECO:0000250|UniProtKB:P48349}. Cytopl...,MENERAKQVYLAKLNEQAERYDEMVEAMKKVAALDVELTIEERNLL...
2,Q96299,14-3-3-like protein GF14 mu,Is associated with a DNA binding complex that ...,Arabidopsis thaliana (Mouse-ear cress).,Nucleus {ECO:0000250|UniProtKB:P48349}. Cytopl...,MGSGKERDTFVYLAKLSEQAERYEEMVESMKSVAKLNVDLTVEERN...
3,Q0VCL1,14-3-3 protein beta/alpha,Adapter protein implicated in the regulation o...,Bos taurus (Bovine).,Cytoplasm {ECO:0000250|UniProtKB:P31946}. Mela...,MTMDKSELVQKAKLAEQAERYDDMAAAMKAVTEQGHELSNEERNLL...
4,P31946,14-3-3 protein beta/alpha,Adapter protein implicated in the regulation o...,Homo sapiens (Human).,Cytoplasm {ECO:0000269|PubMed:17081065}. Melan...,MTMDKSELVQKAKLAEQAERYDDMAAAMKAVTEQGHELSNEERNLL...
...,...,...,...,...,...,...
23627,Q28FD6,Zona pellucida protein D {ECO:0000313|Ensembl:...,NaN,Xenopus tropicalis (Western clawed frog) (Silu...,NaN,MNKMKYYMASWLFVGSILLLTDCISSDLIQQQNAMADLKCDNDQMT...
23628,F6YE22,Mediator of RNA polymerase II transcription su...,NaN,Xenopus tropicalis (Western clawed frog) (Silu...,NaN,MAASLSQELANLLNPQPKFRDPEDDQDEATVARVIDRFEEDGNEDD...
23629,L7N349,High affinity immunoglobulin gamma Fc receptor...,NaN,Xenopus tropicalis (Western clawed frog) (Silu...,NaN,MITGALVRPVVSFSPNWATIAKNDSITLTCTVGPTAPLSQRYSWYR...
23630,F6YR62,Family with sequence similarity 149 member A {...,NaN,Xenopus tropicalis (Western clawed frog) (Silu...,NaN,MPIAAHPYTAWRLKHRSLQQQQRQQHHTLISDTHADLVIIGKGITR...


In [47]:
protein_2_metadata.columns

Index(['Accession', 'Protein Name', 'protein_function', 'organism',
       'subcellular_location', 'sequence'],
      dtype='object')

In [48]:
protein_2_metadata = protein_2_metadata.rename(columns={'Accession':'protein_id', 'Protein Name':'protein_names'})

In [54]:
protein_2_metadata['protein_names'] = protein_2_metadata['protein_names'].str.replace(r'\s?\{.*?\}', '', regex=True).str.strip()
protein_2_metadata['protein_function'] = protein_2_metadata['protein_function'].str.replace(r'\s?\{.*?\}|\s?\((?:Pub).*?\)', '', regex=True).str.strip()
protein_2_metadata['organism'] = protein_2_metadata['organism'].str.replace(r'\.+$', '', regex=True).str.strip()
protein_2_metadata['subcellular_location'] = protein_2_metadata['subcellular_location'].str.replace(r'\s?\{.*?\}|\s?\((?:Pub).*?\)', '', regex=True).str.strip()

In [55]:
protein_2_metadata['protein_names'] = protein_2_metadata['protein_names'].str.replace(r'\s+', ' ', regex=True).str.strip()
protein_2_metadata['protein_function'] = protein_2_metadata['protein_function'].str.replace(r'\s+', ' ', regex=True).str.strip()
protein_2_metadata['organism'] = protein_2_metadata['organism'].str.replace(r'\s+', ' ', regex=True).str.strip()
protein_2_metadata['subcellular_location'] = protein_2_metadata['subcellular_location'].str.replace(r'\s+', ' ', regex=True).str.strip()

In [56]:
protein_2_metadata['protein_names'] = protein_2_metadata['protein_names'].str.replace(r'[.\s]+$', '', regex=True).str.strip()
protein_2_metadata['protein_function'] = protein_2_metadata['protein_function'].str.replace(r'[.\s]+$', '', regex=True).str.strip()
protein_2_metadata['organism'] = protein_2_metadata['organism'].str.replace(r'[.\s]+$', '', regex=True).str.strip()
protein_2_metadata['subcellular_location'] = protein_2_metadata['subcellular_location'].str.replace(r'[.\s]+$', '', regex=True).str.strip()

Adding a column in merged_df which will have the names of the interaction partners proteins. Names are from the uniprot metadata files

In [ ]:
# Map from protein_id to protein_names for lookup
# Convert pandas DataFrame to dict if needed
if isinstance(protein_2_metadata, pd.DataFrame):
    proteinid_to_names = dict(zip(protein_2_metadata['protein_id'], protein_2_metadata['protein_names']))
else:
    # If it's already a Polars DataFrame
    proteinid_to_names = dict(zip(protein_2_metadata['protein_id'].to_list(), protein_2_metadata['protein_names'].to_list()))

def get_partner_names(partners):
    # partners: either None/null or list of uniprot IDs (Polars nulls are None in Python)
    if partners is None:
        return []
    if isinstance(partners, str):
        # Try to parse if stored as a string representation of a list
        import ast
        try:
            partners = ast.literal_eval(partners)
        except Exception:
            partners = partners.split(';') if ';' in partners else [partners]
    # Get names, default to None if missing
    return [proteinid_to_names.get(pid, None) for pid in partners]

# Use Polars map_elements instead of pandas apply
merged_df = merged_df.with_columns(
    pl.col("interaction_partners").map_elements(get_partner_names, return_dtype=pl.List(pl.String)).alias("ppi_formatted")
)


In [58]:
merged_df

protein_id,protein_names,protein_function,organism,subcellular_location,sequence,go_ids,go_bp,go_mf,go_cc,go_bp_created,go_mf_created,go_cc_created,date_added,last_modified,length,structure_path,interpro_ids,interpro_location,string_id,interaction_partners,ppi_formatted
str,str,str,str,str,str,str,str,str,str,f64,f64,f64,str,str,i64,str,list[str],str,str,list[str],list[str]
"""P0DO85""","""Strychnine-10-hydroxylase""","""Monooxygenase involved in the …","""Strychnos nux-vomica (Poison n…","""Membrane; Single-pass membrane…","""MEFSLLYIHTAILGLISLFLILHFVFWRLK…","""['GO:0008150' 'GO:0008152' 'GO…","""['GO:0008150' 'GO:0008152' 'GO…","""['GO:0003674' 'GO:0003824' 'GO…",null,2.0240628e7,2.0240628e7,null,"""2024-11-27""","""2025-06-18""",524,"""AF-P0DO85-F1-model_v6.pdb""","[""IPR050651"", ""IPR017972"", … ""IPR036396""]","""{""IPR036396"": [33, 518], ""IPR0…",null,null,null
"""P0DO87""","""Strychnine-11-hydroxylase""","""Monooxygenase involved in the …","""Strychnos nux-vomica (Poison n…","""Membrane; Single-pass membrane…","""MHSAMSFLLLFSLCFLIHCFVFLLIKKKKA…","""['GO:0008150' 'GO:0008152' 'GO…","""['GO:0008150' 'GO:0008152' 'GO…","""['GO:0003674' 'GO:0003824' 'GO…",null,2.0240628e7,2.0240628e7,null,"""2024-11-27""","""2025-06-18""",508,"""AF-P0DO87-F1-model_v6.pdb""","[""IPR017972"", ""IPR036396"", … ""IPR002401""]","""{""IPR036396"": [23, 500], ""IPR0…",null,null,null
"""P46077""","""14-3-3-like protein GF14 phi""","""Is associated with a DNA bindi…","""Arabidopsis thaliana (Mouse-ea…","""Nucleus. Cytoplasm. Note=Trans…","""MAAPPASSSAREEFVYLAKLAEQAERYEEM…","""['GO:0007154' 'GO:0007165' 'GO…","""['GO:0007154' 'GO:0007165' 'GO…",null,null,2.0080421e7,null,null,"""1995-11-01""","""2025-06-18""",267,"""AF-P46077-F1-model_v6.pdb""","[""IPR023410"", ""IPR036815"", … ""IPR000308""]","""{""IPR036815"": [2, 254], ""IPR00…","""3702.P46077""","[""Q9LK96"", ""Q9LYJ6"", … ""Q9FUT0""]","[""AT3g15090/K15M2_24"", ""Shaggy-related protein kinase epsilon"", … ""Glutathione S-transferase U9""]"
"""P48348""","""14-3-3-like protein G-BOX fact…","""Is associated with a DNA bindi…","""Arabidopsis thaliana (Mouse-ea…","""Nucleus. Cytoplasm. Note=Trans…","""MATTLSRDQYVYMAKLAEQAERYEEMVQFM…","""['GO:0005575' 'GO:0005622' 'GO…",null,null,"""['GO:0005575' 'GO:0005622' 'GO…",null,null,2.0221025e7,"""1996-02-01""","""2025-06-18""",248,"""AF-P48348-F1-model_v6.pdb""","[""IPR023410"", ""IPR036815"", … ""IPR000308""]","""{""IPR036815"": [1, 248], ""IPR00…","""3702.P48348""","[""Q9C9Q2"", ""Q9FZ61"", … ""Q8GUH5""]","[""Protein BRASSINAZOLE-RESISTANT 1"", ""Serine/threonine protein phosphatase 2A 55 kDa regulatory subunit B beta isoform"", … ""non-specific serine/threonine protein kinase""]"
"""Q96299""","""14-3-3-like protein GF14 mu""","""Is associated with a DNA bindi…","""Arabidopsis thaliana (Mouse-ea…","""Nucleus. Cytoplasm. Note=Trans…","""MGSGKERDTFVYLAKLSEQAERYEEMVESM…","""['GO:0007275' 'GO:0008150' 'GO…","""['GO:0007275' 'GO:0008150' 'GO…",null,null,2.0120604e7,null,null,"""1998-07-15""","""2025-06-18""",263,"""AF-Q96299-F1-model_v6.pdb""","[""IPR023410"", ""IPR036815"", … ""IPR000308""]","""{""IPR036815"": [1, 247], ""IPR00…","""3702.Q96299""","[""Q4V397"", ""Q9SRI1"", … ""Q9LZE1""]","[""Ankyrin repeat family protein"", ""Protein transport protein SEC13 homolog A"", … ""Dual specificity phosphatase Cdc25""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""A0A6I8RXM5""","""Mothers against decapentaplegi…",null,"""Xenopus tropicalis (Western cl…","""Cytoplasm. Nucleus""","""MFRTKRSVLVRRLWRSRAPGESGEGEGGER…","""['GO:0001702' 'GO:0003401' 'GO…","""['GO:0001702' 'GO:0003401' 'GO…",null,null,2.0180404e7,null,null,"""2020-12-02""","""2025-10-08""",357,"""AF-A0A6I8RXM5-F1-model_v6.pdb""","[""IPR036578"", ""IPR008984"", … ""IPR003619""]","""{""IPR036578"": [9, 162], ""IPR00…",null,null,null
"""F7EN59""","""Secreted frizzled-related prot…",null,"""Xenopus tropicalis (Western cl…","""Cell membrane; Multi-pass memb…","""MNSEGGIWPLLLFWVTPGILSQAPQASEYD…","""['GO:0008150'

In [60]:
from datasets import Dataset

merged_df_pandas = merged_df.to_pandas()

# Ensure the columns that contain dictionaries or lists are JSON-serializable
temporal_holdout_string = Dataset.from_pandas(merged_df_pandas, preserve_index=False)

In [61]:
from datasets import DatasetDict

temporal_holdout_dataset = DatasetDict({
    "test": temporal_holdout_string,
})

In [62]:
temporal_holdout_dataset

DatasetDict({
    test: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'go_bp_created', 'go_mf_created', 'go_cc_created', 'date_added', 'last_modified', 'length', 'structure_path', 'interpro_ids', 'interpro_location', 'string_id', 'interaction_partners', 'ppi_formatted'],
        num_rows: 17815
    })
})

In [63]:
temporal_holdout_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="temporal_holdout_ext_2022_2025_reasoning",
    commit_message="Upload the Temporal holdout 2022 to 2025 with STRING PPI data"
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/18 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :  15%|#5        | 3.07MB / 19.9MB            

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/14af73e6621e10c580e0aceb6565c77e2d665518', commit_message='Upload the Temporal holdout 2022 to 2025 with STRING PPI data', commit_description='', oid='14af73e6621e10c580e0aceb6565c77e2d665518', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)

## Creating the PPI Metadata file 

In [65]:
filtered_df

protein1,protein2_string,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score,protein2
str,str,i64,i64,i64,i64,i64,i64,i64,i64,str
"""882.DVU_1441""","""882.DVU_0863""",106,0,745,773,148,0,149,955,"""Q72DR6"""
"""882.DVU_1441""","""882.DVU_0862""",106,0,748,769,750,0,149,986,"""Q72DR7"""
"""882.DVU_1441""","""882.DVU_2229""",64,0,655,474,0,0,90,824,"""Q729W8"""
"""882.DVU_1441""","""882.DVU_0410""",97,0,0,794,73,0,0,812,"""Q72F08"""
"""882.DVU_1441""","""882.DVU_0048""",62,0,557,432,0,0,88,755,"""Q72G15"""
…,…,…,…,…,…,…,…,…,…,…
"""1801619.SAMN04515619_11957""","""1801619.SAMN04515619_11956""",804,0,0,0,0,0,0,804,"""A0A1I1NU16"""
"""1801619.SAMN04515619_11957""","""1801619.SAMN04515619_11955""",800,0,0,0,0,0,0,800,"""A0A1I1P074"""
"""1801619.SAMN04515619_11958""","""1801619.SAMN04515619_11956""",804,0,0,0,0,0,0,804,"""A0A1I1NU16"""


In [66]:
protein_2_metadata

,protein_id,protein_names,protein_function,organism,subcellular_location,sequence
0,Q9V648,Guanylate binding protein 128up,Catalyzes the conversion of GTP to GDP through...,Drosophila melanogaster (Fruit fly),NaN,MSTILEKISAIESEMARTQKNKATSAHLGLLKAKLAKLRRELISPK...
1,Q9S9Z8,14-3-3-like protein GF14 omicron,Is associated with a DNA binding complex that ...,Arabidopsis thaliana (Mouse-ear cress),Nucleus. Cytoplasm. Note=Translocates from the...,MENERAKQVYLAKLNEQAERYDEMVEAMKKVAALDVELTIEERNLL...
2,Q96299,14-3-3-like protein GF14 mu,Is associated with a DNA binding complex that ...,Arabidopsis thaliana (Mouse-ear cress),Nucleus. Cytoplasm. Note=Translocates from the...,MGSGKERDTFVYLAKLSEQAERYEEMVESMKSVAKLNVDLTVEERN...
3,Q0VCL1,14-3-3 protein beta/alpha,Adapter protein implicated in the regulation o...,Bos taurus (Bovine),Cytoplasm. Melanosome,MTMDKSELVQKAKLAEQAERYDDMAAAMKAVTEQGHELSNEERNLL...
4,P31946,14-3-3 protein beta/alpha,Adapter protein implicated in the regulation o...,Homo sapiens (Human),Cytoplasm. Melanosome. Note=Identified by mass...,MTMDKSELVQKAKLAEQAERYDDMAAAMKAVTEQGHELSNEERNLL...
...,...,...,...,...,...,...
23627,Q28FD6,Zona pellucida protein D,NaN,Xenopus tropicalis (Western clawed frog) (Silu...,NaN,MNKMKYYMASWLFVGSILLLTDCISSDLIQQQNAMADLKCDNDQMT...
23628,F6YE22,Mediator of RNA polymerase II transcription su...,NaN,Xenopus tropicalis (Western clawed frog) (Silu...,NaN,MAASLSQELANLLNPQPKFRDPEDDQDEATVARVIDRFEEDGNEDD...
23629,L7N349,High affinity immunoglobulin gamma Fc receptor I,NaN,Xenopus tropicalis (Western clawed frog) (Silu...,NaN,MITGALVRPVVSFSPNWATIAKNDSITLTCTVGPTAPLSQRYSWYR...
23630,F6YR62,Family with sequence similarity 149 member A,NaN,Xenopus tropicalis (Western clawed frog) (Silu...,NaN,MPIAAHPYTAWRLKHRSLQQQQRQQHHTLISDTHADLVIIGKGITR...


In [77]:
stringid_to_proteinid = dict(zip(temp_holdout_ext_data['string_id'].to_list(), temp_holdout_ext_data['protein_id'].to_list()))

In [85]:
# Map 'protein1' using stringid_to_proteinid to create 'temp_holdout_protein_id'
filtered_df = filtered_df.with_columns(
    pl.col('protein1').replace(stringid_to_proteinid, default=None).alias('temp_holdout_protein_id')
)

/tmp/ipykernel_1471203/2296454693.py:3: DeprecationWarning: The `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
  pl.col('protein1').replace(stringid_to_proteinid, default=None).alias('temp_holdout_protein_id')


In [86]:
filtered_df

protein1,protein2_string,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score,protein2,temp_holdout_protein_id
str,str,i64,i64,i64,i64,i64,i64,i64,i64,str,str
"""882.DVU_1441""","""882.DVU_0863""",106,0,745,773,148,0,149,955,"""Q72DR6""","""Q72C43"""
"""882.DVU_1441""","""882.DVU_0862""",106,0,748,769,750,0,149,986,"""Q72DR7""","""Q72C43"""
"""882.DVU_1441""","""882.DVU_2229""",64,0,655,474,0,0,90,824,"""Q729W8""","""Q72C43"""
"""882.DVU_1441""","""882.DVU_0410""",97,0,0,794,73,0,0,812,"""Q72F08""","""Q72C43"""
"""882.DVU_1441""","""882.DVU_0048""",62,0,557,432,0,0,88,755,"""Q72G15""","""Q72C43"""
…,…,…,…,…,…,…,…,…,…,…,…
"""1801619.SAMN04515619_11957""","""1801619.SAMN04515619_11956""",804,0,0,0,0,0,0,804,"""A0A1I1NU16""","""A0A1I1NU27"""
"""1801619.SAMN04515619_11957""","""1801619.SAMN04515619_11955""",800,0,0,0,0,0,0,800,"""A0A1I1P074""","""A0A1I1NU27"""
"""1801619.SAMN04515619_11958""","""1801619.SAMN04515619_11956""",804,0,0,0,0,0,0,804,"""A0A1I1NU16""","""A0A1I1P2K9"""


In [89]:
filtered_df = filtered_df[['temp_holdout_protein_id',
'protein1',
'protein2',
 'protein2_string',
 'neighborhood',
 'fusion',
 'cooccurence',
 'coexpression',
 'experimental',
 'database',
 'textmining',
 'combined_score']]

# Rename columns as required
rename_map = {
    'protein2': 'partner_id',
    'protein2_string': 'partner_string_id',
    'protein1': 'temp_holdout_string_id',  # Not present after reordering, but included for completeness
}
# Only rename columns that exist
rename_map = {k: v for k, v in rename_map.items() if k in filtered_df.columns}
filtered_df = filtered_df.rename(rename_map)


In [90]:
filtered_df

temp_holdout_protein_id,temp_holdout_string_id,partner_id,partner_string_id,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score
str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64
"""Q72C43""","""882.DVU_1441""","""Q72DR6""","""882.DVU_0863""",106,0,745,773,148,0,149,955
"""Q72C43""","""882.DVU_1441""","""Q72DR7""","""882.DVU_0862""",106,0,748,769,750,0,149,986
"""Q72C43""","""882.DVU_1441""","""Q729W8""","""882.DVU_2229""",64,0,655,474,0,0,90,824
"""Q72C43""","""882.DVU_1441""","""Q72F08""","""882.DVU_0410""",97,0,0,794,73,0,0,812
"""Q72C43""","""882.DVU_1441""","""Q72G15""","""882.DVU_0048""",62,0,557,432,0,0,88,755
…,…,…,…,…,…,…,…,…,…,…,…
"""A0A1I1NU27""","""1801619.SAMN04515619_11957""","""A0A1I1NU16""","""1801619.SAMN04515619_11956""",804,0,0,0,0,0,0,804
"""A0A1I1NU27""","""1801619.SAMN04515619_11957""","""A0A1I1P074""","""1801619.SAMN04515619_11955""",800,0,0,0,0,0,0,800
"""A0A1I1P2K9""","""1801619.SAMN04515619_11958""","""A0A1I1NU16""","""1801619.SAMN04515619_11956""",804,0,0,0,0,0,0,804


In [98]:
protein_2_metadata[protein_2_metadata['protein_id'].duplicated()].sort_values(by='protein_id')

,protein_id,protein_names,protein_function,organism,subcellular_location,sequence
15733,O23274,Kinesin-like protein KIN-12A,Plus-end directed kinesin-like motor enzyme th...,Arabidopsis thaliana (Mouse-ear cress),"Cytoplasm, cytoskeleton, phragmoplast. Note=Lo...",MKKHFTLPRNAILRDGGEPHSPNPSISKSKPPRKLRSAKENAPPLD...
8550,O23603,NAD-capped RNA hydrolase DXO1,Decapping enzyme for NAD-capped RNAs: specific...,Arabidopsis thaliana (Mouse-ear cress),Cytoplasm. Nucleus,MDSPPKKNPMDDLFGEDSDNDSRSSRSKSSSSGNASSSSSSSSSSS...
23081,O23603,Pentatricopeptide repeat-containing protein At...,NaN,Arabidopsis thaliana (Mouse-ear cress),NaN,MFRKRNLVLESFRRFDSGNVETLISWVLCSRTSKPSLFCTSVKPAR...
35754,O49690,Uncharacterized protein At4g17910,NaN,Arabidopsis thaliana (Mouse-ear cress),Membrane; Multi-pass membrane protein,MTFLLFQLLVLLRYSIGFHCKIDNNGVKSVTSKKNDDEKIVISRNW...
23082,O49690,Putative pentatricopeptide repeat-containing p...,NaN,Arabidopsis thaliana (Mouse-ear cress),NaN,MVRGLVNFTGLSTRLLNICVDSLCKFRKLEKAESLIIDGIRLGVDP...
...,...,...,...,...,...,...
34107,Q9Z0H9,Polyubiquitin-C,[Ubiquitin]: Exists either covalently attached...,Mus musculus (Mouse),[Ubiquitin]: Cytoplasm. Nucleus. Mitochondrion...,MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFA...
35936,Q9ZBC6,Probable multidrug ABC transporter permease YbhR,Part of the ABC transporter complex YbhFSR tha...,Escherichia coli (strain K12),Cell inner membrane; Multi-pass membrane protein,MFHRLWTLIRKELQSLLREPQTRAILILPVLIQVILFPFAATLEVT...
35937,Q9ZBC6,Probable multidrug ABC transporter permease YbhR,Part of the ABC transporter complex YbhFSR tha...,Shigella flexneri,Cell inner membrane; Multi-pass membrane protein,MFHRLWTLIRKELQSLLREPQTRAILILPVLIQVILFPFAATLEVT...
27403,Q9ZL67,DNA-directed RNA polymerase subunit omega,Promotes RNA polymerase assembly. Latches the ...,Helicobacter pylori (strain ATCC 700392 / 2669...,NaN,MKKERTESLVAQALKNIGNDRYMLDNLVFARVKQLNAGAKTLVNMD...


In [95]:
len(protein_2_metadata['protein_id'].unique())

59216

In [107]:
pd.DataFrame(filtered_df['partner_id'].unique().to_pandas())

,partner_id
0,P09383
1,Q7ZSY3
2,Q9NRP2
3,P05738
4,G1T4E6
...,...
62471,Q9H895
62472,Q99NH4
62473,Q9C826
62474,Q9P7L5


In [109]:
temp_metadata = pd.DataFrame(filtered_df['partner_id'].unique().to_pandas()).merge(protein_2_metadata,
    left_on="partner_id",
    right_on="protein_id",
    how="right"
)

In [114]:
filtered_df = filtered_df.to_pandas().merge(
    temp_metadata,
    left_on="partner_id",
    right_on="partner_id",
    how="inner"
)


In [115]:
filtered_df

,temp_holdout_protein_id,temp_holdout_string_id,partner_id,partner_string_id,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score,protein_id,protein_names,protein_function,organism,subcellular_location,sequence
0,Q72C43,882.DVU_1441,Q72DR6,882.DVU_0863,106,0,745,773,148,0,149,955,Q72DR6,Flagellar hook-associated protein 2,Required for morphogenesis and for the elongat...,Nitratidesulfovibrio vulgaris (strain ATCC 295...,Secreted. Bacterial flagellum,MADYLSGSINFTGLGGGVDFAQVIDGLKKIEQVPMNRLTLWKADWQ...
1,Q72C43,882.DVU_1441,Q72DR7,882.DVU_0862,106,0,748,769,750,0,149,986,Q72DR7,Flagellar protein FliS,NaN,Nitratidesulfovibrio vulgaris (strain ATCC 295...,"Cytoplasm, cytosol",MHKAAQAYLQTQVTTTSQGELLLLLYDGAIKFLTQAKERMAARDMA...
2,Q72C43,882.DVU_1441,Q729W8,882.DVU_2229,64,0,655,474,0,0,90,824,Q729W8,Chemotaxis protein MotA,NaN,Nitratidesulfovibrio vulgaris (strain ATCC 295...,Cell membrane; Multi-pass membrane protein. Me...,MDIATFVGVIIGFGLVIGAILMGAAPGKFIDLPSAMIVFGGTFASI...
3,Q72C43,882.DVU_1441,Q72F08,882.DVU_0410,97,0,0,794,73,0,0,812,Q72F08,Phosphate transport regulator,NaN,Nitratidesulfovibrio vulgaris (strain ATCC 295...,NaN,MIVIDGRQTDMKLDSFANLEEILVKVLTEDYLENRIVTDVLVNEET...
4,Q72C43,882.DVU_1441,Q72G15,882.DVU_0048,62,0,557,432,0,0,88,755,Q72G15,Chemotaxis protein MotB,NaN,Nitratidesulfovibrio vulgaris (strain ATCC 295...,Cell membrane; Single-pass membrane protein,MNANPRKTRRKPHQHIPQQGWMLTYSDLVTLLLAFFVLLVSMSSVD...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
192538,A0A1I1NU27,1801619.SAMN04515619_11957,A0A1I1NU16,1801619.SAMN04515619_11956,804,0,0,0,0,0,0,804,A0A1I1NU16,Bacterial deubiquitinase-like protein BilC,Component of the Bil (bacterial ISG15-like) an...,Collimonas sp. (strain OK412),NaN,MNFSIKATIRAWSAPEHKLACRKQLWRKIVAELERRGKRRHEAGTF...
192539,A0A1I1NU27,1801619.SAMN04515619_11957,A0A1I1P074,1801619.SAMN04515619_11955,800,0,0,0,0,0,0,800,A0A1I1P074,Bacterial E1-like protein BilD,Component of the Bil (bacterial ISG15-like) an...,Collimonas sp. (strain OK412),NaN,MNLTLDHTTLHRAAKYFMDSGKAASHEAAMNLLQQFGLTIYVGEEI...
192540,A0A1I1P2K9,1801619.SAMN04515619_11958,A0A1I1NU16,1801619.SAMN04515619_11956,804,0,0,0,0,0,0,804,A0A1I1NU16,Bacterial deubiquitinase-like protein BilC,Component of the Bil (bacterial ISG15-like) an...,Collimonas sp. (strain OK412),NaN,MNFSIKATIRAWSAPEHKLACRKQLWRKIVAELERRGKRRHEAGTF...
192541,A0A1I1P2K9,1801619.SAMN04515619_11958,A0A1I1NU27,1801619.SAMN04515619_11957,804,0,0,0,0,0,0,804,A0A1I1NU27,Bacterial E2-like ubiquitin protein BilB,Component of the Bil (bacterial ISG15-like) an...,Collimonas sp. (strain OK412),NaN,MISAPSPDHLLLEQDLATPEIRCGEIEERWRHIRTSWPHVFFAVSA...


In [116]:
from datasets import Dataset

# Ensure the columns that contain dictionaries or lists are JSON-serializable
temporal_holdout_string_metadata = Dataset.from_pandas(filtered_df, preserve_index=False)

In [118]:
from datasets import DatasetDict

temporal_holdout_string_metadata_dataset = DatasetDict({
    "metadata": temporal_holdout_string_metadata,
})

In [120]:
temporal_holdout_string_metadata_dataset

DatasetDict({
    metadata: Dataset({
        features: ['temp_holdout_protein_id', 'temp_holdout_string_id', 'partner_id', 'partner_string_id', 'neighborhood', 'fusion', 'cooccurence', 'coexpression', 'experimental', 'database', 'textmining', 'combined_score', 'protein_id', 'protein_names', 'protein_function', 'organism', 'subcellular_location', 'sequence'],
        num_rows: 192543
    })
})

In [121]:
temporal_holdout_string_metadata_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="ppi_temp_holdout_ext_22_25_metadata",
    commit_message="Upload the Temporal holdout 2022 to 2025 STRING Metadata"
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/193 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :   4%|3         | 3.77MB / 95.8MB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/7c836aa3e114dfbf308dac3b9cbd555e2e51cdfa', commit_message='Upload the Temporal holdout 2022 to 2025 STRING Metadata', commit_description='', oid='7c836aa3e114dfbf308dac3b9cbd555e2e51cdfa', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)